# Dual-Objective BiLSTM (Presence Head) on Kaggle GPU

Clones `approach/presence-head`. Forked from `approach/full-combo-conv1d`
with one architectural change: a second output head, trained to predict
directly whether each of the 26 letters appears *anywhere* in the word
(a pooled, whole-word prediction), alongside the existing per-position
masked-char head.

Why: the actual guess decision is "does letter X appear anywhere in this
word" -- but every earlier branch answered that by *summing* per-position
softmax scores across all blanks, a heuristic approximation, not
something the model was ever directly trained to get right. The presence
head closes that gap with real supervision (the ground truth is trivially
computable from the training word itself).

Combined loss = masked-char cross-entropy + presence binary cross-entropy.
Everything else (candidate-filtering, n-gram, vowel guard, blend weights)
is identical to `full-combo-conv1d` -- this isolates exactly one variable.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On**.

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/presence-head"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Use the official competition dataset

The cloned repo carries its own copy of train.txt/test.txt (downloaded from
this same competition earlier), but overwrite them here so this notebook
verifiably sources data straight from Kaggle's own `/kaggle/input/`, not an
external GitHub copy -- same content, no ambiguity for anyone reviewing it.

In [ ]:
import shutil
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/train.txt", "train.txt")
shutil.copy("/kaggle/input/competitions/brand-buzzword-hackathon/test.txt", "test.txt")
print("train.txt and test.txt overwritten with the official competition dataset from /kaggle/input/")

## Train

Watch both loss components in the printed output -- `char_loss` should
look similar to the plain full-combo-conv1d run (~2.0 range by epoch 40);
`presence_loss` is new, worth watching that it's actually decreasing too,
not just along for the ride.

In [ ]:
!python src/train_bilstm.py --epochs 40

## Validate

Same held-out-train.txt methodology as every other branch. Compare
directly against `full-combo-conv1d`'s ~53.8-54% (40 epochs) -- this
isolates whether the presence head's direct supervision actually beats
the old per-position-softmax-summing heuristic, since everything else is
identical between the two branches.

In [ ]:
!python src/validate_combined.py --full

## Generate submission.csv

Only run this after confirming the validation number actually beats
full-combo-conv1d.

In [ ]:
!python src/generate_submission_combined.py

## Save outputs

In [ ]:
import shutil
shutil.copy("src/bilstm_dual_head_masker.pt", "/kaggle/working/bilstm_dual_head_masker.pt")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved bilstm_dual_head_masker.pt and submission.csv to /kaggle/working/ -- download from the Output tab")